In [0]:
dbutils.widgets.text("catalog","ai")
dbutils.widgets.text("controlTableSchema","00_etlcontrol")
dbutils.widgets.text("env","dev")
dbutils.widgets.text("source","tutorial")


env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
controlTableSchema = dbutils.widgets.get("controlTableSchema")
source = dbutils.widgets.get("source")


In [0]:
from pyspark.sql.functions import pandas_udf, col,expr, sha2, current_user, current_timestamp, lit, date_format, when, to_json, struct
from typing import Iterator
import pandas as pd
from pyspark.sql.types import ArrayType, StringType, BinaryType, StructType, StructField, LongType, BooleanType
import io, os, sys
from pypdf import PdfReader
from pyspark.sql import SparkSession
import pymupdf

project_root = os.path.abspath("../../") 
if project_root not in sys.path:
    sys.path.append(project_root)

# Reload module to pick up any changes during development
import importlib
import lib.common.sparkConfig
importlib.reload(lib.common.sparkConfig)

import lib.bronze.autoloaderconfig
importlib.reload(lib.bronze.autoloaderconfig)

from lib.common.sparkConfig import SourceInitialize
from lib.bronze.autoloaderconfig import AutoLoaderInitialize



In [0]:
# from pyspark.sql.types import StructType, StructField, LongType, StringType

def getNotebookName():
        current_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return current_notebook.split("/")[-1] if current_notebook.split("/") else current_notebook

spark = SparkSession.builder.appName(getNotebookName()).getOrCreate()
# spark.conf.set("spark.sql.streaming.minBatchesToRetain", "100")

eod_stuct = StructType([
        StructField("isvalid",BooleanType()),
        StructField("error", StringType())
])

class BronzeETL:
        
        def __init__(self, spark, src: SourceInitialize, ldr: AutoLoaderInitialize):
                self.spark = spark
                self.src = src
                self.autoLoaderConfigProperties = ldr.getAutoloaderConfigProperties()
                self.autoLoaderFileMetaProperties = ldr.getAutoloaderFileMetaProperties()
                self.autoloaderTagetTableProperties = ldr.getAutoloaderTagetTableProperties()
                self.autoloaderTagetPartitionColumn = ldr.getAutoloaderTagetPartitionColumn()
                self.sourceProperties = src.getSourceProperties()
                self.metaColumns = self.getFileMetaData()
                self.sourceName = src.getSourceName()
                self.loadid = src.getLoadId()
                self.rejectedRecordsTable = src.getRejectedRecordsTable()
                self.inline_threshold_bytes = self.sourceProperties.get("INLINE_THRESHOLD_BYTES")

        def getFileMetaData(self):

                return [f'{val} as {name}' for name,val in self.autoLoaderFileMetaProperties.items()]

        
        @staticmethod
        def applyFileRouting(batch_df):

                return (batch_df
                # .drop("content")
                # .withColumnRenamed("file_content","content")
                .select("*",sha2(col("content"), 256).alias("documentKey"))
                )
                
        def invalid_file_write(self, batch_df, batch_id):
                
                
                invalid_batch_df = batch_df.filter(col("isvalid") != True)

                (invalid_batch_df.drop("content").select(lit("batch_id").alias("batch_id"),
                                to_json(struct("*")).alias("rejected_payload"),
                                lit(self.sourceName).alias("source_system"),
                                current_timestamp().alias("rejected_at"),
                                current_user().alias("createdby"),
                                )
                        .write
                        .format("delta")
                        .mode("append")
                        .option("mergeSchema", "true")
                        .saveAsTable(f'{self.autoloaderTagetTableProperties.get("targetCatelog")}.{self.autoloaderTagetTableProperties.get("targetSchema")}.{self.autoloaderTagetTableProperties.get("layer")}_{self.rejectedRecordsTable}')
                        )
                
                return batch_df.filter(col("isvalid") == "VALID FILE")
                
                
        @staticmethod
        def end_of_file_validation(content) -> tuple:

                if b'%%EOF' in content:
                        return (True,{"isvalid":True, "error":""})
                else:
                        return (False,{"isvalid":False, "error":"END OF FILE NOT FOUND"})
                
        @staticmethod
        def Zero_pages_file_open_validation(content) -> tuple:
                try: 
                        with pymupdf.open(stream= io.BytesIO(content)) as docs:
                                if len(docs.pages) == 0:
                                        return (False,{"isvalid":False, "error":"ZERO PAGES"})
                                else:
                                        return (True,{"isvalid":True, "error":""})
                except Exception as e:
                        return (False,{"isvalid":False, "error":str(e)})


        @staticmethod
        def isEncrepted(content) -> tuple:
                try:
                        with pymupdf.open(stream= io.BytesIO(content)) as docs:
                                if docs.is_encrypted:
                                        return (False,{"isvalid":False, "error":"FILE IS ENCRYPTED"})
                                else:
                                        return (True,{"isvalid":True, "error":""})
                except Exception as e:
                        return (False,{"isvalid":False, "error":str(e)})
        
        @staticmethod
        def file_size_validation(length) -> tuple:

                if length > 0:
                        return (True,{"isvalid":True, "error":""})
                else:
                        return (False,{"isvalid":False, "error":"ZERO SIZE FILE"})
                

        @staticmethod
        def validate_pdf(content, length) -> dict:

                isvalid = True
                error = []
                encrept=BronzeETL.isEncrepted(content)
                isvalid = encrept[0]
                error.append(encrept[1])

                eof = BronzeETL.end_of_file_validation(content)
                isvalid = eof[0]
                error.append(eof[1])

                zero_page_file = BronzeETL.Zero_pages_file_open_validation(content)
                isvalid = zero_page_file[0]    
                error.append(zero_page_file[1])

                file_size = BronzeETL.file_size_validation(length)
                isvalid = file_size[0]    
                error.append(file_size[1])

                return {
                        "isvalid": isvalid,
                        "error": error
                }
                

        @staticmethod
        def validate_pdf_batch(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:

                for batch in iterator:

                        isvalid = []
                        error = []

                        for _, row in batch.iterrows():

                                content = row["content"]
                                length = row["length"]
                                validation = BronzeETL.validate_pdf(content, length)
                                isvalid.append(validation["isvalid"])
                                error.append(validation["error"])

                        batch["isvalid"] = isvalid
                        batch["error"] = error

                        yield batch
                
                        


        
        def processBatch(self,batch_df, batch_id):

                # batch_df.persist()
                # spark = batch_df.sparkSession

                if batch_df.isEmpty():
                        return
                targetProps = self.autoloaderTagetTableProperties
                cnt = batch_df.select("path").count()                
                batch_df = batch_df.select("*", lit(batch_id).alias("batchid"), lit(self.loadid).alias("loadid"))
                batch_df = BronzeETL.applyFileRouting(batch_df)
                batch_schema = batch_df.schema
                schema = (batch_schema.add( StructField("isvalid", BooleanType(), True)) 
                                        .add( StructField("error", ArrayType(StructType([StructField("isvalid", BooleanType()), StructField("error", StringType())])), True)))
                batch_df = batch_df.mapInPandas(BronzeETL.validate_pdf_batch, schema=schema)
                # display(batch_df)
                
                (batch_df.write
                        .format(f'{targetProps.get("targetType")}')
                        .mode(f'{targetProps.get("mode")}')
                        .option("mergeSchema", "true")
                        .saveAsTable(f'{targetProps.get("targetCatelog")}.{targetProps.get("targetSchema")}.{targetProps.get("layer")}_{ self.sourceName}_document')
                                
                        )
                
                # Audit insert inlined (SourceInitialize holds a SparkSession and
                # cannot be pickled into the foreachBatch worker). Types match
                # the existing bronze_auditlog table (legacy string columns).
                audit_spark = batch_df.sparkSession
                audit_records = [(
                                self.loadid,
                                str(batch_id),
                                self.sourceName,
                                str(cnt),
                                "SUCCESS",
                                None
                                )]

                audit_schema = StructType([
                        StructField("loadid", StringType(), False),
                        StructField("batchid", StringType(), False),
                        StructField("sourcename", StringType(), False),
                        StructField("files_processed", StringType(), False),
                        StructField("status", StringType(), False),
                        StructField("error_message", StringType(), True)
                ])

                (audit_spark.createDataFrame(audit_records, schema=audit_schema)
                        .select("*", current_timestamp().alias("loaddt" ), current_user().alias("createdby"))
                        .write.format("delta")
                        .mode("append")
                        .option("mergeSchema", "true")
                        .saveAsTable(f'{targetProps.get("targetCatelog")}.{targetProps.get("targetSchema")}.{targetProps.get("layer")}_auditlog')
                )
                
                # batch_df.unpersist()    

        def bronzeLoad(self):

                
                query = (self.spark.readStream.format("cloudFiles") 
                 .options(**self.autoLoaderConfigProperties)
                 .load(self.sourceProperties.get("file_path"))
                 .selectExpr("*", *self.metaColumns)
                 .writeStream
                 .foreachBatch(self.processBatch)
                 .option("checkpointLocation", f"{self.sourceProperties.get('checkpint')}/{self.sourceName}")
                 .trigger(availableNow=True)
                 .start()
                )

                query.awaitTermination()
                

                
                
if __name__ == "__main__":



        print(source)
        src = SourceInitialize(spark, getNotebookName(), env, catalog, controlTableSchema, source)
        src.checkSetup()

        ldr = AutoLoaderInitialize(spark, catalog, controlTableSchema,src.getAutoloaderPropertiesTable(), src.getSourceId())
        ldr.checkSetup()

        bz_etl = BronzeETL(spark, src, ldr)        
        bz_etl.bronzeLoad()

In [0]:
df = (spark.read.table("ai.`02_bronze`.bronze_tutorial_document"))

display(df.drop("content"))

df = spark.read.table("ai.`02_bronze`.bronze_auditlog")
display(df)

